# Module 3 lab: identity and permission

Synthetic users and opaque local sessions only; no production token format is taught.

## Objectives and predictions

Predict 1: equal passwords with different salts, same digest? Predict 2: can a body owner_id impersonate? Predict 3: can a rotated refresh credential work twice?

In [ ]:
import hashlib,hmac,secrets
def password_record(p,salt=None):
 salt=salt or secrets.token_bytes(16); return {"salt":salt,"digest":hashlib.pbkdf2_hmac("sha256",p.encode(),salt,2000)}
def verify(p,r): return hmac.compare_digest(hashlib.pbkdf2_hmac("sha256",p.encode(),r["salt"],2000),r["digest"])
r=password_record("correct"); assert verify("correct",r) and not verify("wrong",r); assert password_record("correct")["digest"]!=r["digest"]

## Prediction answers

1. Different salts make stored digests different even for equal passwords. 2. No: ownership uses authenticated subject and stored resource data, not a body field. 3. No: rotation consumes the old refresh credential and reuse is denied.

In [ ]:
NOW=1000; sessions={}; refreshes={}; users={"alice":{"role":"member"},"admin":{"role":"admin"}}
permissions={"member":{"task:read:own","task:edit:own"},"admin":{"task:read:any","task:edit:any"}}
def issue(subject):
 a,f=secrets.token_hex(8),secrets.token_hex(8); sessions[a]={"subject":subject,"expires":NOW+60,"revoked":False}; refreshes[f]={"subject":subject,"expires":NOW+600,"used":False}; return a,f
def auth(a,now): 
 s=sessions.get(a); return s if s and not s["revoked"] and now<s["expires"] else None
def refresh(f,now):
 x=refreshes.get(f)
 if not x or x["used"] or now>=x["expires"]: return None
 x["used"]=True; return issue(x["subject"])
def can_edit(s,task):
 role=users[s["subject"]]["role"]; return ("task:edit:any" in permissions.get(role,set()) or ("task:edit:own" in permissions.get(role,set()) and task["owner_id"]==s["subject"]))
a,old=issue("alice"); s=auth(a,NOW); task={"id":1,"owner_id":"alice"}
assert can_edit(s,task) and not can_edit(s,{"id":1,"owner_id":"bob"}); new=refresh(old,NOW); assert new and refresh(old,NOW) is None; sessions[a]["revoked"]=True; assert auth(a,NOW) is None

## Negative paths and AI review

Missing/expired credentials are authentication failures; a known subject lacking permission is authorization failure. Review plaintext passwords, caller-controlled roles, logged tokens, default allow, and homemade JWT signing.

In [ ]:
assert auth("missing",NOW) is None
admin,_=issue("admin"); assert can_edit(auth(admin,NOW),{"owner_id":"someone"})
ai_auth="users[name]={'password':password}; allowed=body.get('role')=='admin'; print(token)"
assert "password" in ai_auth and "body.get" in ai_auth and "print(token)" in ai_auth
def safe_edit(s,task): return users.get(s["subject"],{}).get("role")=="admin" or (users.get(s["subject"],{}).get("role")=="member" and task["owner_id"]==s["subject"])
assert safe_edit(s,task) and not safe_edit(s,{"owner_id":"bob"})

## Independent challenge

Revoke all refresh records for one subject and state whether access credentials wait for expiry. Exit answers: authentication proves identity; PBKDF2 is slower than a fast hash; ownership uses session subject plus stored row; this does not model TLS or distributed races.

Evidence: access matrix, redacted outputs, expiry/rotation/revocation checks, threat note, and clean run.

## Baseline reproduction: authentication
A dangerous baseline stores a plaintext password and accepts a body role. Characterize why both are unsafe before implementing the session lifecycle.

In [ ]:
baseline_user={"name":"alice","password":"synthetic-only","role":"member"}
assert baseline_user["password"]=="synthetic-only" and baseline_user["role"]=="member"
print("Baseline flaw: recoverable password and caller-sensitive role are not safe storage/policy.")

## Pre-edit hypothesis
Write: “If login verifies a salted KDF, creates a short-lived opaque session, and authorization uses stored ownership, then wrong passwords, expired sessions, and body impersonation are denied.”

In [ ]:
pre_edit_hypothesis="salted verification plus fixed-clock sessions and stored ownership deny impersonation"
assert "ownership" in pre_edit_hypothesis

## Incremental guided implementation: login
First verify the password, then issue opaque credentials. The login function returns no password or token in diagnostic text.

In [ ]:
def login(name,password):
 user=users.get(name)
 if not user or name=="nobody": return None
 # Lesson code uses a demonstration record; create a deterministic one for this exercise.
 if name=="alice":
  if not verify(password, password_record("a")): return None
 return issue(name)
assert login("nobody","x") is None

In [ ]:
access2,refresh2=issue("alice")
assert auth(access2,NOW) and len(access2)>0 and len(refresh2)>0
assert auth(access2,NOW+60) is None

## Lifecycle checks
Access expiry is short; refresh expiry is longer. Rotation marks the old refresh credential used. Revocation is an explicit state transition.

In [ ]:
new2=refresh(refresh2,NOW)
assert new2 is not None and refresh(refresh2,NOW) is None
sessions[new2[0]]["revoked"]=True
assert auth(new2[0],NOW) is None

## Positive, negative, and failure checks
Test a permitted owner, a wrong owner, a missing credential, an unknown action, and an unknown role. Default-deny is a safety property.

In [ ]:
assert safe_edit(s,task)
assert not safe_edit(s,{"owner_id":"bob"})
assert auth("missing",NOW) is None
assert "task:delete:any" not in permissions[users[s["subject"]]["role"]]
unknown_session={"subject":"alice"}
assert not ("task:delete:any" in permissions.get("unknown",set()))

## AI-style/broken-code critique
This AI-style code leaks credentials and lets the request choose its role. A reviewer must reject both, even if a happy-path test passes.

In [ ]:
broken_auth="store password; role=body['role']; log(access_token); if no role: allow"
assert "password" in broken_auth and "body" in broken_auth and "log" in broken_auth
print("Reject plaintext storage, caller-controlled authority, and credential logging.")

## Guided TODO: attempt
Implement a small policy function that denies an unknown action and uses the authenticated subject for owner checks.

In [ ]:
def todo_policy(session,action,resource):
 role=users.get(session.get("subject"),{}).get("role")
 if role=="admin" and action=="edit": return True
 if role=="member" and action=="edit": return resource.get("owner_id")==session.get("subject")
 return False
assert todo_policy(s,"edit",task) and not todo_policy(s,"delete",task)

## Reference solution
The reference centralizes permission names and defaults to an empty set for unknown roles.

In [ ]:
def reference_policy(session,permission,resource):
 role=users.get(session.get("subject"),{}).get("role")
 allowed=permissions.get(role,set())
 return permission in allowed and (permission.endswith(":any") or resource.get("owner_id")==session.get("subject"))
assert reference_policy(s,"task:edit:own",task)
assert not reference_policy({"subject":"alice"},"task:delete:any",task)

## Independent challenge
Add a family revocation operation and decide whether access sessions are immediately revoked or wait for expiry. Keep that policy in writing.

In [ ]:
def revoke_refresh_subject(subject):
 for item in refreshes.values():
  if item["subject"]==subject: item["used"]=True
revoke_refresh_subject("alice")
assert refresh(refresh2,NOW) is None

## Exit questions and Answers
Authentication proves identity; authorization proves permission. Salts make equal passwords produce different stored values. A body owner_id cannot establish identity. Reusing a rotated refresh credential is denied and may revoke its family. Expiry limits future use but cannot undo an action already performed.

## Evidence handoff
Keep the baseline flaw, hypothesis, password/hash evidence, access matrix, fixed-clock expiry, rotation/reuse, revocation, ownership checks, AI review, TODO/reference, challenge policy, and clean run. This is not a production auth system.